# Remove background for CBIS-DDSM dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import cv2
import shutil
import pandas as pd

# Check the quantity of dataset for each path.

In [6]:
malignant_path = "/content/drive/MyDrive/DDSMset/MALIGNANT/"
benign_path = "/content/drive/MyDrive/DDSMset/BENIGN/"

print("Malignant images:", len(os.listdir(malignant_path)))
print("Benign images:", len(os.listdir(benign_path)))

Malignant images: 784
Benign images: 772


**Define path for dataset**

In [8]:
base_dataset = "/content/drive/MyDrive/DDSMset"

# PREPROCESSING

#REMOVE BACKGROUND PREPROCESSING


**PROCESS**


Remove the black background by detecting the main breast region and keeping only the largest foreground contour in the mammogram image.

This function preprocesses each mammogram by converting it to grayscale, detecting the main foreground area and masking out unnecessary background regions.



In [9]:
def medical_prep_clean(image_path):
    img = cv2.imread(image_path)
    if img is None: return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest_cnt = max(contours, key=cv2.contourArea)
        mask = np.zeros_like(gray)
        cv2.drawContours(mask, [largest_cnt], -1, 255, -1)

        cleaned = cv2.bitwise_and(img, img, mask=mask)
        return cleaned
    return img

source_base = "/content/drive/MyDrive/DDSMset"
output_base = "/content/drive/MyDrive/DDSMset_Cleaned"
classes = ['BENIGN', 'MALIGNANT']

for cls in classes:
    os.makedirs(os.path.join(output_base, cls), exist_ok=True)
    path = os.path.join(source_base, cls)
    files = os.listdir(path)

    print(f"Processing {cls}...")
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            cleaned_img = medical_prep_clean(os.path.join(path, f))
            if cleaned_img is not None:
                cv2.imwrite(os.path.join(output_base, cls, f), cleaned_img)

print("Background Removal Complete!")

Processing BENIGN...
Processing MALIGNANT...
Background Removal Complete!


**RESULT**

Read all images from the source dataset folders (DDSMset), apply background cleaning and save the processed images into new output folders (DDSMset_Cleaned).


**CHECK THE SIZE OF DATASET**

Checking the size of DDSMset dataset same with DDSMset_Cleaned.

In [10]:
malignant_path = "/content/drive/MyDrive/DDSMset_Cleaned/MALIGNANT/"
benign_path = "/content/drive/MyDrive/DDSMset_Cleaned/BENIGN/"

print("Malignant images:", len(os.listdir(malignant_path)))
print("Benign images:", len(os.listdir(benign_path)))

Malignant images: 784
Benign images: 772
